# Point-BERT LiDAR Tokenization (nuScenes)
## With 3D Interactive Plotly Visualizations

This notebook:
- Loads nuScenes LiDAR
- Creates Point-BERT style patches (FPS + kNN)
- Visualizes raw point cloud
- Visualizes patch centers
- Visualizes token-colored patches
- Interactive 3D using Plotly


In [1]:
!pip -q install nuscenes-devkit plotly
import os
import numpy as np
import torch
import plotly.graph_objects as go
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.data_classes import LidarPointCloud


ModuleNotFoundError: No module named 'torch'

## Load nuScenes LiDAR Frame

In [ ]:
DATAROOT = '/content'  # Change path
VERSION = 'v1.0-mini'

nusc = NuScenes(version=VERSION, dataroot=DATAROOT, verbose=False)
sample = nusc.sample[0]
lidar_token = sample['data']['LIDAR_TOP']
sd = nusc.get('sample_data', lidar_token)
lidar_path = os.path.join(nusc.dataroot, sd['filename'])

pc = LidarPointCloud.from_file(lidar_path)
points = pc.points[:3].T  # (N,3)

print('Raw LiDAR shape:', points.shape)

## Interactive 3D Visualization of Raw LiDAR

In [ ]:
def plot_lidar(points, title='LiDAR 3D View'):
    fig = go.Figure(data=[go.Scatter3d(
        x=points[:,0],
        y=points[:,1],
        z=points[:,2],
        mode='markers',
        marker=dict(size=1)
    )])
    fig.update_layout(title=title, scene_aspectmode='data')
    fig.show()

plot_lidar(points)

## FPS + kNN Patch Creation

In [ ]:
def farthest_point_sampling(xyz, n_samples):
    xyz = torch.tensor(xyz, dtype=torch.float32)
    N = xyz.shape[0]
    centroids = torch.zeros(n_samples, dtype=torch.long)
    distance = torch.ones(N) * 1e10
    farthest = torch.randint(0, N, (1,)).item()
    for i in range(n_samples):
        centroids[i] = farthest
        centroid = xyz[farthest]
        dist = torch.sum((xyz - centroid) ** 2, dim=1)
        distance = torch.minimum(distance, dist)
        farthest = torch.argmax(distance).item()
    return centroids

def knn_indices(xyz, centers, k):
    xyz = torch.tensor(xyz, dtype=torch.float32)
    centers = torch.tensor(centers, dtype=torch.float32)
    d = torch.cdist(centers, xyz)
    idx = torch.topk(d, k=k, largest=False).indices
    return idx

N_POINTS = 1024
NUM_GROUP = 64
GROUP_SIZE = 32

sampled_idx = np.random.choice(points.shape[0], N_POINTS, replace=False)
sampled_points = points[sampled_idx]

fps_idx = farthest_point_sampling(sampled_points, NUM_GROUP)
centers = sampled_points[fps_idx]

nn_idx = knn_indices(sampled_points, centers, GROUP_SIZE)
patches = sampled_points[nn_idx]

print('Patches shape:', patches.shape)

## Visualize Patch Centers

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=centers[:,0],
    y=centers[:,1],
    z=centers[:,2],
    mode='markers',
    marker=dict(size=5)
)])
fig.update_layout(title='Patch Centers', scene_aspectmode='data')
fig.show()

## Color Patches (Simulated Token IDs)

In [ ]:
token_ids = np.random.randint(0, 20, NUM_GROUP)
all_patch_points = patches.reshape(-1,3)
colors = np.repeat(token_ids, GROUP_SIZE)

fig = go.Figure(data=[go.Scatter3d(
    x=all_patch_points[:,0],
    y=all_patch_points[:,1],
    z=all_patch_points[:,2],
    mode='markers',
    marker=dict(size=2, color=colors, colorscale='Viridis')
)])
fig.update_layout(title='Token Colored Patches', scene_aspectmode='data')
fig.show()